# Session 2: two readings, and how wrong can the height be?
**A. Real-world problem.** A moat, a fence or a river stops you reaching the base of the tower,
so you do not know the distance `d`. And your phone clinometer is only good to about half a
degree. Where should you stand so that a 30 m tower is measured to within 0.5 m?

**Plan.** (1) height from two angles at two spots, (2) propagate the angle error into a height
error, (3) turn the error formula into a design rule.

In [ ]:
import pathlib
import math

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
EYE = 1.6


def cot(deg):
    return 1.0 / math.tan(math.radians(deg))

## Step 1: two angles, one baseline
Read angle `a1` at a far spot, walk `b` metres straight towards the tower, read `a2`. With
`x` the unknown horizontal distance from the near spot to the base, and H the height above eye:

    x = H cot(a2)   and   x + b = H cot(a1)   so   H = b / (cot(a1) - cot(a2))

In [ ]:
def height_two_angles(b, a1_deg, a2_deg, eye=EYE):
    """Total height (m). a1 is the FAR reading (smaller), a2 the NEAR reading (larger), b the walk (m)."""
    if not 0 < a1_deg < a2_deg < 90:
        raise ValueError("need 0 < a1 < a2 < 90 degrees (far reading smaller than near reading)")
    return b / (cot(a1_deg) - cot(a2_deg)) + eye


def distance_to_base(b, a1_deg, a2_deg):
    """Horizontal distance (m) from the near spot to the base, found as a by-product."""
    return b * cot(a2_deg) / (cot(a1_deg) - cot(a2_deg))

## Step 2: error propagation
If the true height above the eye is H = d tan(theta), a reading error `dtheta` (in radians)
changes it by about  dH = d sec^2(theta) dtheta,  and since d = H / tan(theta):

    dH = 2 H dtheta / sin(2 theta)

This is smallest at theta = 45 degrees (sin 2theta = 1) and grows fast at small and large angles.

In [ ]:
def height_error(d, angle_deg, sigma_deg):
    """First-order height error (m): d * sec^2(theta) * sigma, with sigma converted to radians."""
    return d / math.cos(math.radians(angle_deg)) ** 2 * math.radians(sigma_deg)


def best_window(h_above_eye, sigma_deg, tol):
    """Distances (m) between which the first-order error stays within tol, or None if never.

    From 2 H sigma / sin(2 theta) <= tol:  sin(2 theta) >= 2 H sigma / tol.
    """
    s = 2 * h_above_eye * math.radians(sigma_deg) / tol
    if s > 1 + 1e-12:
        return None
    s = min(s, 1.0)
    th_lo = 0.5 * math.asin(s)
    th_hi = math.pi / 2 - th_lo
    return h_above_eye / math.tan(th_hi), h_above_eye / math.tan(th_lo)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    H = 28.4          # 30 m tower, eye at 1.6 m
    sig = 0.5         # degrees
    print("window for 0.5 m:", best_window(H, sig, 0.5))
    for d in (10, 20, 28.4, 40, 60, 100):
        th = math.degrees(math.atan(H / d))
        print(f"d = {d:5.1f} m  angle {th:5.1f} deg  error {height_error(d, th, sig):.3f} m")

    d = np.linspace(5, 100, 300)
    th = np.degrees(np.arctan(H / d))
    err = d / np.cos(np.radians(th)) ** 2 * math.radians(sig)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(d, err, label="height error d sec^2(theta) sigma")
    ax.axhline(0.5, color="red", ls="--", label="0.5 m target")
    lo, hi = best_window(H, sig, 0.5)
    ax.axvspan(lo, hi, color="green", alpha=0.2, label=f"stand {lo:.1f} to {hi:.1f} m away")
    ax.set_ylim(0, 3)
    ax.set_xlabel("distance from the base (m)")
    ax.set_ylabel("height error (m)")
    ax.set_title("Where to stand: 30 m tower, reading good to 0.5 degrees")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "error_vs_distance.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.plot([0, 40], [0, 0], "k-")
    ax2.plot([0, 0], [0, 30], lw=4, color="tab:blue", label="tower 30 m")
    for x, name in ((40, "far spot, a1"), (30, "near spot, a2")):
        ax2.plot([x, 0], [1.6, 30], "--", color="orange")
        ax2.plot([x], [1.6], "ro")
        ax2.annotate(name, (x, 4), ha="center", fontsize=8)
    ax2.set_xlabel("distance from the tower base (m)")
    ax2.set_ylabel("height (m)")
    ax2.set_title("Two readings, walk b = 10 m towards the tower")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "two_angle_geometry.png", dpi=120)

## Where this stops
These error formulas assume a small, random reading error and a level surface. Session 3 tests
them with a simulation, adds a hill, and shows what happens near 90 degrees.

## Coding tasks
1. **Easy.** Use `height_two_angles` with b = 10, a1 = 30, a2 = 40 and eye 1.6 and print the height and `distance_to_base`; check the distance with a tangent by hand.
2. **Stretch.** Use `best_window` for sigma = 0.25 degrees and a limit of 0.5 m for the 30 m tower and describe how the window changes.
3. **Explain.** Explain why the two-angle formula needs the walk to be reasonably long compared with the distance (try b = 1 m in the code with readings rounded to 0.5 degrees).

## Assessment
- **A. Conceptual.** A friend says: stand as far away as possible, then the angle is small and the error is small. Is that right for a tower of a given height? What is true and what is not?
- **B. Numerical.** A 20 m tower (18.4 m above your eye) is read from 18.4 m and from 50 m, with a reading error of 0.5 degrees. Find the height error at each distance.
- **C. Graph and simulation interpretation.** On the error graph for the 30 m tower (0.5 degrees), where is the minimum and what is its value? Is a 0.5 m limit reachable at 60 m?
- **D. Design and optimisation challenge.** Design: the 30 m tower must be measured to within 0.5 m with a clinometer good to 0.5 degrees, but a wall means you can only stand 40 m or farther away. What can you do? Then suppose the clinometer is only good to 1 degree: what is the best possible single-reading error and what could you change?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    H = 28.4
    a1 = math.degrees(math.atan(H / 40))
    a2 = math.degrees(math.atan(H / 30))
    checks.append(("exact readings recover the height: two angles at 40 m and 30 m give 30 m",
                   abs(L.height_two_angles(10, a1, a2, 1.6) - 30.0) < 1e-9))
    h = L.height_two_angles(10, 35.4, 43.5, 1.6)
    checks.append((f"worked example: b = 10 m, 35.4 and 43.5 degrees, eye 1.6 m gives 29.90 m (got {h:.2f})", abs(h - 29.900) < 0.01))
    checks.append(("distance to the base is found too: 30 m for the exact readings", abs(L.distance_to_base(10, a1, a2) - 30.0) < 1e-9))
    checks.append(("hand calculation: 45 degrees at 28.4 m, sigma 0.5 degrees, error = 2 H sigma = 0.496 m",
                   abs(L.height_error(28.4, 45, 0.5) - 2 * 28.4 * math.radians(0.5)) < 1e-9))
    e20, e45, e70 = (L.height_error(H / math.tan(math.radians(t)), t, 0.5) for t in (20, 45, 70))
    checks.append(("error is smallest at 45 degrees", e45 < e20 and e45 < e70))
    w = L.best_window(H, 0.5, 0.5)
    checks.append((f"worked example: design window is 24.9 to 32.4 m (got {w[0]:.1f} to {w[1]:.1f})",
                   abs(w[0] - 24.9) < 0.06 and abs(w[1] - 32.4) < 0.06))
    checks.append(("error equals the target at both edges of the window",
                   all(abs(L.height_error(d, math.degrees(math.atan(H / d)), 0.5) - 0.5) < 1e-6 for d in w)))
    checks.append(("edge: a 0.3 m target cannot be met with a 0.5 degree reading (window is None)", L.best_window(H, 0.5, 0.3) is None))
    edge = L.best_window(H, 0.5, 2 * H * math.radians(0.5))
    checks.append(("edge: a target equal to the 45 degree error gives a single point at d = H",
                   edge is not None and abs(edge[0] - H) < 1e-6 and abs(edge[1] - H) < 1e-6))
    for a, b in ((40, 30), (30, 30), (0, 30), (30, 90)):
        try:
            L.height_two_angles(10, a, b)
            ok = False
        except ValueError:
            ok = True
        checks.append((f"edge: readings ({a}, {b}) are rejected", ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
